# Session 12: Optimisation in practice (Python lab)

You will break a softmax on purpose and fix it by subtracting the maximum, compute a log-sum-exp, meet the limits of 16-bit numbers, and fit a degree 9 curve to 10 points with a ridge (L2) penalty for several values of lambda.

**How to work.** In each step: read the idea, write a PREDICT guess, run the cell, READ the output, then CHANGE one thing in the experiment cell. Cells with `None  # your code here` are yours to fill. The automatic checks print OK or NOT YET. Run the cells from the top, in order. Step 1 prints a RuntimeWarning and nan on purpose.

---
## Step 0: Set up

numpy for numbers, matplotlib for the picture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("ready")

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: Break the naive softmax on purpose

softmax(z)_i = exp(z_i) / sum of exp(z_j). The scores (logits) are 1000, 1001 and 1002. A float64 number cannot be larger than about 1.8e308, and exp(710) already exceeds that.

**PREDICT first.** What will the naive softmax of (1000, 1001, 1002) print? Numbers, zeros, inf or nan?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
z = np.array([1000.0, 1001.0, 1002.0])
naive = np.exp(z) / np.exp(z).sum()      # exp overflows to inf, then inf / inf
print(np.exp(z), naive)

**READ the output.** What is exp(1000) here, and why is the answer nan? (Read the warning too.)

> My answer: 

In [ ]:
check("the naive softmax is nan, as expected", float(np.isnan(naive).all()), 1, 0)

**CHANGE one thing.** Run the experiment cell: exp(709) and exp(710). Where does float64 give up?

> What I saw: 

In [ ]:
print(np.exp(709.0), np.exp(710.0))

---
## Step 2: Fix it: subtract the maximum, then log-sum-exp

Subtracting the same constant c from every score does not change the softmax, because e^(-c) cancels. Choose c = the maximum so that every exponent is at most 0. Log-sum-exp is m + ln(sum of e^(z - m)).

**PREDICT first.** What will the stable softmax of (1000, 1001, 1002) be? (The slides say it equals softmax(0, 1, 2).)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: one line per idea. The slide "Read the stable softmax before you write it" shows the pattern.
m = None  # your code here: the largest score
e = None  # your code here: the exponentials of the shifted scores
stable = None  # your code here: each exponential divided by their total
lse = None  # your code here: the log-sum-exp
logprobs = None  # your code here: the log of each probability, without forming the probability
small = None  # your code here: the naive softmax of [2.0, 1.0, 0.1]
check("stable softmax of (1000, 1001, 1002)", stable, [0.09003057317038046, 0.24472847105479764, 0.6652409557748218], 0.0001)
check("log-sum-exp", lse, 1002.4076059644444, 0.0001)
check("log-probabilities equal the log of the stable softmax", logprobs, None if stable is None else np.log(stable), 0.0001)
check("softmax(2, 1, 0.1) with the naive way", small, [0.6590011388859679, 0.2424329707047139, 0.09856589040931818], 0.0001)

**READ the output.** The stable result is the same as softmax(0, 1, 2). Why do only the differences between scores matter?

> My answer: 

**CHANGE one thing.** Run the experiment cell: shift the scores by 5000 instead. Predict first whether the stable softmax changes.

> What I saw: 

In [ ]:
z2 = np.array([6000.0, 6001.0, 6002.0])
e2 = np.exp(z2 - z2.max())
print(np.round(e2 / e2.sum(), 4))

---
## Step 3: Number formats and a long product

float16 stores a number in 16 bits: its largest value is 65504. A product of many probabilities falls below the smallest float64 number (about 5e-324) and becomes exactly 0 from n = 324 tokens on. A sum of logs does not.

**PREDICT first.** What is exp(12) in float16? What is the product 0.1 to the power 400 in float64, and the sum of 400 copies of ln(0.1)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: np.finfo describes a number format. Wrap single numbers in np.float16(...) to compute in float16.
f16_max = None  # your code here: the largest float16 number
e12 = None  # your code here: the exponential of 12 in float16 (put it inside "with np.errstate(over=\"ignore\"):" to hide the warning)
lost = None  # your code here: 1 plus 0.0004, both as float16
prod400 = None  # your code here: the product of 400 probabilities of 0.1
sumlog400 = None  # your code here: the sum of the logs of 400 probabilities of 0.1
check("largest float16 number", f16_max, 65504, 0)
check("exp(12) in float16 is inf", None if e12 is None else float(np.isinf(e12)), 1, 0)
check("1 + 0.0004 in float16 is still 1", None if lost is None else float(lost), 1, 0)
check("the product of 400 probabilities of 0.1 is 0", prod400, 0, 0)
check("the sum of 400 logs of 0.1", sumlog400, -921.034, 0.001)

**READ the output.** Why is the update of 0.0004 lost? How many significant digits does float16 keep?

> My answer: 

**CHANGE one thing.** Run the experiment cell: the product and the sum of logs for 300 tokens. Is the product still above zero?

> What I saw: 

In [ ]:
print(0.1 ** 300, round(300 * np.log(0.1), 3))

---
## Step 4: Ridge regression in closed form

Ten training points, a degree 9 polynomial: columns 1, x, x^2, ..., x^9. The ridge answer is w = (X^T X + lambda P)^-1 X^T y, with P the identity matrix whose first entry is 0, so the intercept is not penalised. Illustrative data (made-up noisy points from a smooth curve).

**PREDICT first.** Without a penalty the curve passes through every point. Will its weights be small or enormous? Which lambda do you expect to give the lowest error on 20 new points?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
xtr = np.linspace(-1, 1, 10)
ytr = np.array([-0.65, -0.53, -0.55, -0.53, -0.06, -0.05, 0.61, 0.78, 1.26, 1.46])
xte = np.linspace(-0.97, 0.97, 20)
yte = np.array([-0.73, -0.58, -0.63, -0.76, -0.32, -0.43, -0.64, -0.5, -0.17, -0.03, -0.03, 0.42, 0.41, 0.43, 0.66, 0.85, 0.78, 1.02, 0.99, 1.33])
Xtr = np.vander(xtr, 10, increasing=True)     # columns 1, x, ..., x^9
Xte = np.vander(xte, 10, increasing=True)
P = np.eye(10)
P[0, 0] = 0.0                                 # do not penalise the intercept
print(Xtr.shape, Xte.shape)

In [ ]:
# Hint: the closed form is on the slide "Ridge has a closed-form answer". np.linalg.solve is better than an explicit inverse. RMSE is defined on the ridge table slide. The intercept is not in the sum of squared weights.
def ridge(X, y, lam):
    if lam == 0:
        return np.linalg.lstsq(X, y, rcond=None)[0]
    return None  # your code here: the ridge weights for lam > 0
def rmse(X, w, y):
    return None  # your code here: the root of the mean squared error
w0, w1, w10 = ridge(Xtr, ytr, 0.0), ridge(Xtr, ytr, 0.1), ridge(Xtr, ytr, 10.0)
norm0 = norm1 = norm10 = None  # your code here: the sum of squared weights without the intercept
check("sum of squared weights, lambda = 0.1", norm1, 1.5473232439900502, 0.0001)
check("sum of squared weights, lambda = 10", norm10, 0.11806834041917336, 0.0001)
check("the unpenalised weights are enormous", None if norm0 is None else float(norm0 > 10000), 1, 0)
check("training error, lambda = 0.1", None if w1 is None else rmse(Xtr, w1, ytr), 0.10915869386278189, 0.0001)
check("error on the 20 new points, lambda = 0", None if w0 is None else rmse(Xte, w0, yte), 0.6656913035625057, 0.001)
check("error on the 20 new points, lambda = 0.1", None if w1 is None else rmse(Xte, w1, yte), 0.15844652630242717, 0.0001)

**READ the output.** The unpenalised training error is almost 0 but its error on new points is the largest. What do we call that? What happens to the training error as lambda grows?

> My answer: 

**CHANGE one thing.** Run the experiment cell: the error on the 20 new points for six values of lambda. Which lambda is lowest?

> What I saw: 

In [ ]:
for lam in [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]:
    w = ridge(Xtr, ytr, lam)
    print(lam, round(rmse(Xtr, w, ytr), 4), round(rmse(Xte, w, yte), 4))

In [ ]:
xg = np.linspace(-1, 1, 200)
if w1 is None:
    print("Fill in the ridge answer cell above first.")
else:
    plt.figure(figsize=(6, 3))
    plt.scatter(xtr, ytr, color="black", zorder=3)
    plt.plot(xg, np.sin(1.5 * xg) + 0.4 * xg ** 2, "k--", label="true curve")
    plt.plot(xg, np.vander(xg, 10, increasing=True) @ w1, color="#AE2525", label="ridge, lambda = 0.1")
    plt.legend()
    plt.show()

---
## Step 5: Weight decay, a constraint and clipping

Weight decay: w <- (1 - 2 eta lambda) w - eta dL/dw. A constraint: the lowest loss w1^2 + w2^2 on the line w1 + w2 = 2. Clipping: g <- g c / (length of g) when the length exceeds c.

**PREDICT first.** One step with w = 2, eta = 0.1, lambda = 0.5 and gradient 1: bigger or smaller than 2? Where on the line is the loss lowest?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: the weight-decay slide gives the update, and the constraint slide gives the loss. To clip, rescale the vector to length 5.
eta, lam, g = 0.1, 0.5, 1.0
w_new = None  # your code here: the new w after one decay step, starting from w = 2
w1v = np.linspace(0, 2, 5)
loss = None  # your code here: the loss at each point of w1v
best = None  # your code here: the w1 with the lowest loss
gvec = np.array([30.0, 40.0])
clipped = None  # your code here: gvec shortened to length 5, same direction
check("weight after one step with decay", w_new, 1.7, 0.0001)
check("lowest loss on the line", None if loss is None else loss.min(), 2, 0.0001)
check("best w1 on the line", best, 1, 0.0001)
check("clipped gradient", clipped, [3, 4], 0.0001)

**READ the output.** At the best point, the loss slope (2 w1, 2 w2) = (2, 2) is parallel to the line slope (1, 1). Why does that tell us it is the best point?

> My answer: 

**CHANGE one thing.** Run the experiment cell: take lambda = 2 and keep the other numbers. Predict whether w_new is below or above 1.7.

> What I saw: 

In [ ]:
print((1 - 2 * 0.1 * 2.0) * 2.0 - 0.1 * 1.0)

---
## Step 6: Show the checks

Run this last cell. A learner who has filled every answer sees "All ... checks pass".

In [ ]:
show_checks()